## Train a character-level GPT on some text data

The inputs here are simple text files, which we chop up to individual characters and then train GPT on. So you could say this is a char-transformer instead of a char-rnn. Doesn't quite roll off the tongue as well. In this example we will feed it some Shakespeare, which we'll get it to predict character-level.

In [6]:
# set up logging
import logging
logging.basicConfig(
        format="%(asctime)s - %(levelname)s - %(name)s -   %(message)s",
        datefmt="%m/%d/%Y %H:%M:%S",
        level=logging.INFO,
)

In [7]:
# make deterministic
from mingpt.utils import set_seed
set_seed(42)

In [8]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

In [9]:
%%writefile dataset.py

import math
import torch
from torch.utils.data import Dataset

class CharDataset(Dataset):

    def __init__(self, data, block_size):
        chars = sorted(list(set(data)))
        data_size, vocab_size = len(data), len(chars)
        print('data has %d characters, %d unique.' % (data_size, vocab_size))
        
        self.stoi = { ch:i for i,ch in enumerate(chars) }
        self.itos = { i:ch for i,ch in enumerate(chars) }
        self.block_size = block_size
        self.vocab_size = vocab_size
        self.data = data
    
    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # grab a chunk of (block_size + 1) characters from the data
        chunk = self.data[idx:idx + self.block_size + 1]
        # encode every character to an integer
        dix = [self.stoi[s] for s in chunk]
        """
        arrange data and targets so that the first i elements of x
        will be asked to predict the i-th element of y. Notice that
        the eventual language model will actually make block_size
        individual predictions at the same time based on this data,
        so we are being clever and amortizing the cost of the forward
        pass of the network. So for example if block_size is 4, then
        we could e.g. sample a chunk of text "hello", the integers in
        x will correspond to "hell" and in y will be "ello". This will
        then actually "multitask" 4 separate examples at the same time
        in the language model:
        - given just "h", please predict "e" as next
        - given "he" please predict "l" next
        - given "hel" predict "l" next
        - given "hell" predict "o" next
        
        In addition, because the DataLoader will create batches of examples,
        every forward/backward pass during traning will simultaneously train
        a LOT of predictions, amortizing a lot of computation. In particular,
        for a batched input of integers X (B, T) where B is batch size and
        T is block_size and Y (B, T), the network will during training be
        simultaneously training to make B*T predictions, all at once! Of course,
        at test time we can paralellize across batch B, but unlike during training
        we cannot parallelize across the time dimension T - we have to run
        a forward pass of the network to recover the next single character of the 
        sequence along each batch dimension, and repeatedly always feed in a next
        character to get the next one.
        
        So yes there is a big asymmetry between train/test time of autoregressive
        models. During training we can go B*T at a time with every forward pass,
        but during test time we can only go B at a time, T times, with T forward 
        passes.
        """
        x = torch.tensor(dix[:-1], dtype=torch.long)
        y = torch.tensor(dix[1:], dtype=torch.long)
        return x, y


Overwriting dataset.py


In [10]:
block_size = 64 # spatial extent of the model for its context (reduced from 128 to fit in ~4 GB GPU memory; attention cost is O(T^2))


In [11]:
!curl https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -o input.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1089k  100 1089k    0     0   670k      0  0:00:01  0:00:01 --:--:--  670k


In [12]:
# you can download this file at https://github.com/karpathy/char-rnn/blob/master/data/tinyshakespeare/input.txt
from dataset import CharDataset
text = open('input.txt', 'r').read() # don't worry we won't run out of file handles
train_dataset = CharDataset(text, block_size) # one line of poem is roughly 50 characters

data has 1115394 characters, 65 unique.


In [13]:
from mingpt.model import GPT, GPTConfig
mconf = GPTConfig(train_dataset.vocab_size, train_dataset.block_size,
                  n_layer=8, n_head=8, n_embd=512)
model = GPT(mconf)

08/17/2026 12:48:37 - INFO - mingpt.model -   number of parameters: 2.531942e+07


In [14]:
from mingpt.trainer import Trainer, TrainerConfig

# initialize a trainer instance and kick off training
# NOTE: to fit in ~4 GB of GPU memory we use block_size=64, a smaller per-step batch (16)
# with 4x gradient accumulation (effective batch stays 64) plus fp16 mixed precision (halves
# activations). Restart the kernel first if you previously hit a CUDA OOM - the GPU memory
# cached by the failed run is still held by the old kernel until it is restarted.
tconf = TrainerConfig(max_epochs=2, batch_size=16, learning_rate=6e-4,
                      lr_decay=True, warmup_tokens=512*20, final_tokens=2*len(train_dataset)*block_size,
                      num_workers=4, gradient_accumulation_steps=4, amp=True)
trainer = Trainer(model, train_dataset, None, tconf)
trainer.train()

epoch 1 iter 69707: train loss 0.53028. lr 3.000346e-04: 100%|██████████| 69709/69709 [41:22<00:00, 28.08it/s]
epoch 2 iter 69707: train loss 0.24575. lr 6.000000e-05: 100%|██████████| 69709/69709 [41:31<00:00, 27.98it/s]


In [15]:
# alright, let's sample some character-level Shakespeare
from mingpt.utils import sample

context = "O God, O God!"
x = torch.tensor([train_dataset.stoi[s] for s in context], dtype=torch.long)[None,...].to(trainer.device)
y = sample(model, x, 2000, temperature=1.0, sample=True, top_k=10)[0]
completion = ''.join([train_dataset.itos[int(i)] for i in y])
print(completion)

O God, O God! that e'er this tongue of mine,
That laid the sentence of dread banishment
On yon proud man, should take it off again
With words of sooth! O that I were as great
As is my grief, or lesser than my name!
Or that I could forget what I have been,
Or not remember what I must be now!
Swell'st thou, proud heart? I'll give thee scope to beat,
Since foes have scope to beat both thee and me.

DUKE OF AUMERLE:
Then give me leave that I may turn the key,
That no man enter till my tale be done.

HENRY BOLINGBROKE:
Have thy desire.

DUKE OF YORK:

HENRY BOLINGBROKE:
What is the matter, uncle? speak;
Recover breath; tell us how near is danger,
That we may arm us to encounter it.

DUKE OF YORK:
Peruse this writing here, and thou shalt know
The treason that my haste forbids me show.

DUKE OF AUMERLE:
Remember, as thou read'st, thy promise pass'd:
I do repent me; repeal daily I have begun,
For sorrow ends not when it seemeth done.
Commend me to thy brother: soon at night
I'll send him certa

In [16]:
# well that was fun